In [2]:
from __future__ import print_function

import math
import numpy as np
import numpy.linalg as nla
import pandas as pd
from pathlib import Path
import re
import six
import duckdb
from os.path import join
import tensorflow as tf
from matplotlib import pyplot as plt
from sklearn.model_selection import train_test_split
from keras_tuner import HyperParameters

In [3]:
current_dir = Path.cwd()
data_dir = current_dir.joinpath("data")
print(data_dir)

income_df_raw = pd.read_csv(data_dir.joinpath("ACSST5Y2023.S1901-Data.csv"), sep=",", encoding='latin-1') #S1901 Income in the Past 12 Months (ACS 5-Year Estimate - In 2023 Inflation-Adjusted Dollars)() - found here: https://data.census.gov/table/ACSST5Y2023.S1901?t=Income+and+Poverty&g=010XX00US$8600000&y=2023
business_df_raw = pd.read_csv(data_dir.joinpath("zbp23detail.zip"), sep=",", encoding='latin-1') #Business Employment Data - found here: https://www2.census.gov/programs-surveys/cbp/datasets/2023/zbp23detail.zip
iou_df_raw = pd.read_csv(data_dir.joinpath("iou_zipcodes_2023.csv"), sep=",", encoding='latin-1') #Investor Owned Utilities - found here: https://data.openei.org/files/6225/iou_zipcodes_2023.csv
non_iou_df_raw = pd.read_csv(data_dir.joinpath("non_iou_zipcodes_2023.csv"), sep=",", encoding='latin-1') #Non Investor Owned Utilities - found here: https://data.openei.org/files/6225/non_iou_zipcodes_2023.csv


def print_df_detail(dataframe,df_name = 'Dataframe'):
	print(f"{df_name} shape: {dataframe.shape}")
	index_length = len(str(len(dataframe.columns)))
	values_length = len(str(len(dataframe)))
	columns_width = max(len(str(column)) for column in dataframe.columns)
	
	for index, column in enumerate(dataframe.columns):
		print(f"Column {index:0{index_length}d}: {column:>{columns_width}}  │  {len(dataframe[column].unique()):{values_length}d} unique values  │  dtype:{str(dataframe[column].dtype):>8}  │  nulls:{dataframe[column].isna().sum():{values_length}d}  │  zeroes:{(dataframe[column] == 0).sum():{values_length}d}")

	display(dataframe.head(5))

c:\Users\Roe 2019\Documents\GitHub\watts-the-cost-DATASCI-207\data


C:\Users\Roe 2019\AppData\Local\Temp\ipykernel_60064\1053968025.py:5: DtypeWarning: Columns (2,3,34,35,66,67,98,99) have mixed types. Specify dtype option on import or set low_memory=False.
  income_df_raw = pd.read_csv(data_dir.joinpath("ACSST5Y2023.S1901-Data.csv"), sep=",", encoding='latin-1') #S1901 Income in the Past 12 Months (ACS 5-Year Estimate - In 2023 Inflation-Adjusted Dollars)() - found here: https://data.census.gov/table/ACSST5Y2023.S1901?t=Income+and+Poverty&g=010XX00US$8600000&y=2023


In [4]:
################
#Clean income_df
################

#Make copies to play with
income_df = income_df_raw.copy()

#promote first row to headers

#set headers to first row
income_df.columns = income_df.iloc[0]

#Keep everything from the second row onwards
income_df = income_df[1:]

#Reset the row numbers (index) back to starting at 0
income_df.reset_index(drop=True, inplace=True)

#Clear the name of the columns axis (removes the leftover '0' index label)
income_df.columns.name = None

print_df_detail(income_df, "income_df")







income_df shape: (33772, 131)
Column 000:                                                                                           Geography  │  33772 unique values  │  dtype:  object  │  nulls:    0  │  zeroes:    0
Column 001:                                                                                Geographic Area Name  │  33772 unique values  │  dtype:  object  │  nulls:    0  │  zeroes:    0
Column 002:                                                                         Estimate!!Households!!Total  │  12980 unique values  │  dtype:  object  │  nulls:    0  │  zeroes:  838
Column 003:                                                                  Margin of Error!!Households!!Total  │   1936 unique values  │  dtype:  object  │  nulls:    0  │  zeroes:    0
Column 004:                                                      Estimate!!Households!!Total!!Less than $10,000  │    462 unique values  │  dtype:  object  │  nulls:    0  │  zeroes:    0
Column 005:                   

,Geography,Geographic Area Name,Estimate!!Households!!Total,Margin of Error!!Households!!Total,"Estimate!!Households!!Total!!Less than $10,000","Margin of Error!!Households!!Total!!Less than $10,000","Estimate!!Households!!Total!!$10,000 to $14,999","Margin of Error!!Households!!Total!!$10,000 to $14,999","Estimate!!Households!!Total!!$15,000 to $24,999","Margin of Error!!Households!!Total!!$15,000 to $24,999",...,Margin of Error!!Nonfamily households!!Median income (dollars),Estimate!!Nonfamily households!!Mean income (dollars),Margin of Error!!Nonfamily households!!Mean income (dollars),Estimate!!Nonfamily households!!PERCENT ALLOCATED!!Household income in the past 12 months,Margin of Error!!Nonfamily households!!PERCENT ALLOCATED!!Household income in the past 12 months,Estimate!!Nonfamily households!!PERCENT ALLOCATED!!Family income in the past 12 months,Margin of Error!!Nonfamily households!!PERCENT ALLOCATED!!Family income in the past 12 months,Estimate!!Nonfamily households!!PERCENT ALLOCATED!!Nonfamily income in the past 12 months,Margin of Error!!Nonfamily households!!PERCENT ALLOCATED!!Nonfamily income in the past 12 months,NaN
0,860Z200US00601,ZCTA5 00601,5611,258,24.0,3.2,15.8,3.6,25.7,4.1,...,1623,17435,2864,(X),(X),(X),(X),14.1,(X),NaN
1,860Z200US00602,ZCTA5 00602,12546,510,22.6,2.9,11.8,2.0,20.7,2.5,...,1833,17806,2205,(X),(X),(X),(X),19.4,(X),NaN
2,860Z200US00603,ZCTA5 00603,19537,671,28.5,2.5,12.2,1.9,18.0,1.9,...,1199,18433,2062,(X),(X),(X),(X),35.3,(X),NaN
3,860Z200US00606,ZCTA5 00606,1871,192,23.6,5.4,16.6,6.1,21.2,5.8,...,4018,18057,3134,(X),(X),(X),(X),11.0,(X),NaN
4,860Z200US00610,ZCTA5 00610,8838,459,18.7,2.9,12.5,3.0,21.9,3.4,...,2299,19469,2506,(X),(X),(X),(X),11.5,(X),NaN


In [5]:
income_df_short = income_df.iloc[:,:29]

#Gather ZIP from Geo Name
income_df_short.insert(0,'zip',income_df_short['Geographic Area Name'].astype(str).str[-5:])

#Drop Geo and Geo Name
income_df_short.drop(columns=['Geography','Geographic Area Name'], axis=1, inplace=True)

rename_substring_map ={
    'Estimate':'',
	'Margin of Error':'moe',
	'Total':'',
    '!!':'_',
	'(dollars)':'',
    ' ':'_',
	',999':'k',
	',000':'k'
	
    
}
rename_dict = {}
for column in income_df_short.columns.copy():
	new_column_name = column
	for old_substr, new_substr in rename_substring_map.items():
		if old_substr in new_column_name:
			new_column_name = new_column_name.replace(old_substr, new_substr)
	while '__' in new_column_name:
		new_column_name = new_column_name.replace('__','_')		
	new_column_name = new_column_name.strip('_')
	rename_dict[column] = new_column_name.lower()


income_df_short.rename(columns=rename_dict,inplace=True)

income_df_short = income_df_short.apply(pd.to_numeric, errors='coerce')


income_df_final = income_df_short.dropna()

print_df_detail(income_df_final)




Dataframe shape: (30507, 28)
Column 00:                                                                 zip  │  30507 unique values  │  dtype:   int64  │  nulls:    0  │  zeroes:    0
Column 01:                                                          households  │  10947 unique values  │  dtype:   int64  │  nulls:    0  │  zeroes:    0
Column 02:                                                      moe_households  │   1146 unique values  │  dtype:   int64  │  nulls:    0  │  zeroes:    0
Column 03:                                           households_less_than_$10k  │    347 unique values  │  dtype: float64  │  nulls:    0  │  zeroes: 3177
Column 04:                                       moe_households_less_than_$10k  │    610 unique values  │  dtype: float64  │  nulls:    0  │  zeroes:    0
Column 05:                                             households_$10k_to_$14k  │    337 unique values  │  dtype: float64  │  nulls:    0  │  zeroes: 3975
Column 06:                               

,zip,households,moe_households,households_less_than_$10k,moe_households_less_than_$10k,households_$10k_to_$14k,moe_households_$10k_to_$14k,households_$15k_to_$24k,moe_households_$15k_to_$24k,households_$25k_to_$34k,...,moe_households_$100k_to_$149k,households_$150k_to_$199k,moe_households_$150k_to_$199k,households_$200k_or_more,moe_households_$200k_or_more,households_median_income,moe_households_median_income,households_mean_income,moe_households_mean_income,households_percent_allocated_household_income_in_the_past_12_months
0,601,5611,258,24.0,3.2,15.8,3.6,25.7,4.1,12.7,...,1.1,0.0,0.9,0.3,0.5,18571.0,1365.0,24781.0,1987.0,15.4
1,602,12546,510,22.6,2.9,11.8,2.0,20.7,2.5,15.6,...,0.8,0.7,0.5,0.4,0.4,21702.0,2026.0,30378.0,2097.0,20.6
2,603,19537,671,28.5,2.5,12.2,1.9,18.0,1.9,10.4,...,1.0,1.6,0.7,0.8,0.4,19243.0,1444.0,32478.0,1777.0,44.3
3,606,1871,192,23.6,5.4,16.6,6.1,21.2,5.8,18.2,...,0.7,0.0,2.8,0.0,2.8,20226.0,3619.0,25125.0,3029.0,17.3
4,610,8838,459,18.7,2.9,12.5,3.0,21.9,3.4,15.1,...,0.9,0.5,0.5,1.9,1.5,23732.0,1614.0,34513.0,4140.0,18.6


In [6]:
################
#Clean business_df
################

#Make copies to play with
business_df = business_df_raw.copy()

#Gather Business Sector from NAICS
business_df.insert(1,'sector',business_df['naics'].astype(str).str[:2])

#Remove ZIP name, city, city_name, state abbrev, NAICS
business_df.drop(columns=['name','city','stabbr','cty_name','naics'], axis=1, inplace=True)



#replace missing nujmbers with 0
business_df = business_df.apply(pd.to_numeric, errors='coerce')
business_df = business_df.fillna(0)


print_df_detail(business_df, "business_df")


business_df shape: (2974116, 12)
Column 00:      zip  │    34954 unique values  │  dtype:   int64  │  nulls:      0  │  zeroes:      0
Column 01:   sector  │       25 unique values  │  dtype: float64  │  nulls:      0  │  zeroes:  34954
Column 02:      est  │     2048 unique values  │  dtype:   int64  │  nulls:      0  │  zeroes:      0
Column 03:      n<5  │     1338 unique values  │  dtype: float64  │  nulls:      0  │  zeroes:1157095
Column 04:     n5_9  │      498 unique values  │  dtype: float64  │  nulls:      0  │  zeroes:2299985
Column 05:   n10_19  │      402 unique values  │  dtype: float64  │  nulls:      0  │  zeroes:2514368
Column 06:   n20_49  │      340 unique values  │  dtype: float64  │  nulls:      0  │  zeroes:2662743
Column 07:   n50_99  │      171 unique values  │  dtype: float64  │  nulls:      0  │  zeroes:2872839
Column 08: n100_249  │      134 unique values  │  dtype: float64  │  nulls:      0  │  zeroes:2922921
Column 09: n250_499  │       66 unique values  │ 

,zip,sector,est,n<5,n5_9,n10_19,n20_49,n50_99,n100_249,n250_499,n500_999,n1000
0,501,0.0,5,5.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
1,1001,0.0,460,217.0,79.0,73.0,53.0,24.0,13.0,0.0,0.0,0.0
2,1001,22.0,3,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
3,1001,22.0,3,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
4,1001,23.0,49,32.0,8.0,3.0,3.0,0.0,0.0,0.0,0.0,0.0


In [7]:
#####################################################
#USE THIS FOR ALL SECTORS SUMMED - Results in 11 columns
#####################################################
business_df_final = business_df.drop(columns=['sector'],axis=1).groupby(['zip'], as_index=False).sum()


print_df_detail(business_df_final, "business_df_final")

business_df_final shape: (34954, 11)
Column 00:      zip  │  34954 unique values  │  dtype:   int64  │  nulls:    0  │  zeroes:    0
Column 01:      est  │   6305 unique values  │  dtype:   int64  │  nulls:    0  │  zeroes:    0
Column 02:      n<5  │   4335 unique values  │  dtype: float64  │  nulls:    0  │  zeroes: 2389
Column 03:     n5_9  │   1755 unique values  │  dtype: float64  │  nulls:    0  │  zeroes:10229
Column 04:   n10_19  │   1379 unique values  │  dtype: float64  │  nulls:    0  │  zeroes:13906
Column 05:   n20_49  │   1116 unique values  │  dtype: float64  │  nulls:    0  │  zeroes:17113
Column 06:   n50_99  │    476 unique values  │  dtype: float64  │  nulls:    0  │  zeroes:22628
Column 07: n100_249  │    318 unique values  │  dtype: float64  │  nulls:    0  │  zeroes:25242
Column 08: n250_499  │    121 unique values  │  dtype: float64  │  nulls:    0  │  zeroes:30481
Column 09: n500_999  │     62 unique values  │  dtype: float64  │  nulls:    0  │  zeroes:33538
Col

,zip,est,n<5,n5_9,n10_19,n20_49,n50_99,n100_249,n250_499,n500_999,n1000
0,501,5,5.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
1,1001,2300,978.0,254.0,226.0,183.0,50.0,17.0,0.0,0.0,0.0
2,1002,2701,1271.0,438.0,333.0,145.0,25.0,0.0,0.0,0.0,0.0
3,1003,62,20.0,21.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
4,1004,5,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0


In [8]:
#####################################################
#USE THIS FOR DATA BY SECTOR - Results in 75 columns
#####################################################


#business_df2 = business_df.groupby(['zip','sector'], as_index=False).sum()
#business_df2.insert(10,'n<500',business_df2.iloc[:,3:10].sum(axis=1))
#business_df2.drop(columns=business_df2.columns[3:10], inplace=True)


#business_df_melted = business_df2.melt(
#	id_vars =['zip','sector'],
#	value_vars = ['n<500','n500_999','n1000'],
#	var_name ='size_class',
#    value_name ='estabs'
#)

#business_df_melted['sector_size'] = business_df_melted['sector'].astype(int).astype(str) + ' ' + business_df_melted['size_class'].astype(str)

#business_df_final2 = business_df_melted.pivot_table(
#	index = 'zip',
#	columns = 'sector_size',
#	values ='estabs',
#	aggfunc = 'sum',
#	fill_value=0
#)

#business_df_final.reset_index()
#business_df_final.columns.name = None

#print_df_detail(business_df_final2, "business_df_final2")







In [9]:
################
#Clean iou_df
################

#Make copies to play with
iou_df = iou_df_raw.copy()

print_df_detail(iou_df, "iou_df")

iou_df shape: (52074, 9)
Column 0:          zip  │  31542 unique values  │  dtype:   int64  │  nulls:    0  │  zeroes:    0
Column 1:        eiaid  │    142 unique values  │  dtype:   int64  │  nulls:    0  │  zeroes:    0
Column 2: utility_name  │    142 unique values  │  dtype:  object  │  nulls:    0  │  zeroes:    0
Column 3:        state  │     50 unique values  │  dtype:  object  │  nulls:    0  │  zeroes:    0
Column 4: service_type  │      2 unique values  │  dtype:  object  │  nulls:    0  │  zeroes:    0
Column 5:    ownership  │      1 unique values  │  dtype:  object  │  nulls:    0  │  zeroes:    0
Column 6:    comm_rate  │    251 unique values  │  dtype: float64  │  nulls:    0  │  zeroes:    0
Column 7:     ind_rate  │    239 unique values  │  dtype: float64  │  nulls:    0  │  zeroes:  458
Column 8:     res_rate  │    241 unique values  │  dtype: float64  │  nulls:    0  │  zeroes: 2637


,zip,eiaid,utility_name,state,service_type,ownership,comm_rate,ind_rate,res_rate
0,85321,176,Ajo Improvement Co,AZ,Bundled,Investor Owned,0.103993,0.000000,0.117085
1,36560,195,Alabama Power Co,AL,Bundled,Investor Owned,0.140915,0.076707,0.159023
2,36513,195,Alabama Power Co,AL,Bundled,Investor Owned,0.140915,0.076707,0.159023
3,36280,195,Alabama Power Co,AL,Bundled,Investor Owned,0.140915,0.076707,0.159023
4,35473,195,Alabama Power Co,AL,Bundled,Investor Owned,0.140915,0.076707,0.159023


In [10]:
################
#Clean non_iou_df
################

#Make copies to play with
non_iou_df = non_iou_df_raw.copy()

print_df_detail(non_iou_df, "non_iou_df")

non_iou_df shape: (28068, 9)
Column 0:          zip  │  19259 unique values  │  dtype:   int64  │  nulls:    0  │  zeroes:    0
Column 1:        eiaid  │   1060 unique values  │  dtype:   int64  │  nulls:    0  │  zeroes:    0
Column 2: utility_name  │   1060 unique values  │  dtype:  object  │  nulls:    0  │  zeroes:    0
Column 3:        state  │     49 unique values  │  dtype:  object  │  nulls:    0  │  zeroes:    0
Column 4: service_type  │      3 unique values  │  dtype:  object  │  nulls:    0  │  zeroes:    0
Column 5:    ownership  │      6 unique values  │  dtype:  object  │  nulls:    0  │  zeroes:    0
Column 6:    comm_rate  │   1146 unique values  │  dtype: float64  │  nulls:    0  │  zeroes:  113
Column 7:     ind_rate  │    998 unique values  │  dtype: float64  │  nulls:    0  │  zeroes: 2673
Column 8:     res_rate  │   1145 unique values  │  dtype: float64  │  nulls:    0  │  zeroes: 1871


,zip,eiaid,utility_name,state,service_type,ownership,comm_rate,ind_rate,res_rate
0,39730,55,City of Aberdeen - (MS),MS,Bundled,Municipal,0.12279,0.056408,0.123420
1,38858,55,City of Aberdeen - (MS),MS,Bundled,Municipal,0.12279,0.056408,0.123420
2,21864,84,A & N Electric Coop,MD,Bundled,Cooperative,0.14049,0.000000,0.138776
3,21824,84,A & N Electric Coop,MD,Bundled,Cooperative,0.14049,0.000000,0.138776
4,21866,84,A & N Electric Coop,MD,Bundled,Cooperative,0.14049,0.000000,0.138776


In [21]:
inc_cols = 'inc."' +'",\n		inc."'.join(income_df_final.columns) + '"'
bs_cols = 'bs."' +'",\n		bs."'.join(business_df_final.drop(columns='zip').columns) + '"'


SQL = f"""
	WITH
    rate_type_key (rate_type, rate_type_index) as (
		values
			('commercial',0),
            ('industrial',1),
            ('residential',2),
    
    
    
    ),
    Utility_Data AS (
		SELECT
        iou.zip,
		iou.service_type,
        iou.ownership,
        rtk.rate_type_index,
        case
        	when rtk.rate_type = 'commercial' then iou.comm_rate
        	when rtk.rate_type = 'industrial' then iou.ind_rate
            when rtk.rate_type = 'residential' then iou.res_rate
            end AS rate

        FROM iou_df iou, rate_type_key rtk
        
        
        
        UNION ALL
	
		SELECT
        non_iou.zip,
		non_iou.service_type,
        non_iou.ownership,
		rtk.rate_type_index,
        case
        	when rtk.rate_type = 'commercial' then non_iou.comm_rate
        	when rtk.rate_type = 'industrial' then non_iou.ind_rate
            when rtk.rate_type = 'residential' then non_iou.res_rate
            end AS rate

        FROM non_iou_df non_iou, rate_type_key rtk
    ),
    
    
    
    COMBINED_DATA AS (
		SELECT
		{inc_cols},
		{bs_cols},
		ud.service_type,
		ud.ownership,
        ud.rate_type_index,
        ud.rate
		
    
    
    
    	FROM income_df_final inc

		INNER JOIN business_df_final bs
		ON inc.zip = bs.zip
       
	   	INNER JOIN Utility_Data ud
	   	on ud.zip = inc.zip
           
           
        WHERE ud.rate is not null and ud.rate > 0
    )

    SELECT *
    
    
    FROM combined_data CD
    
    
"""

print(SQL)


df_clean = duckdb.sql(SQL).df()

display(df_clean.head(5))


	WITH
    rate_type_key (rate_type, rate_type_index) as (
		values
			('commercial',0),
            ('industrial',1),
            ('residential',2),



    ),
    Utility_Data AS (
		SELECT
        iou.zip,
		iou.service_type,
        iou.ownership,
        rtk.rate_type_index,
        case
        	when rtk.rate_type = 'commercial' then iou.comm_rate
        	when rtk.rate_type = 'industrial' then iou.ind_rate
            when rtk.rate_type = 'residential' then iou.res_rate
            end AS rate

        FROM iou_df iou, rate_type_key rtk



        UNION ALL

		SELECT
        non_iou.zip,
		non_iou.service_type,
        non_iou.ownership,
		rtk.rate_type_index,
        case
        	when rtk.rate_type = 'commercial' then non_iou.comm_rate
        	when rtk.rate_type = 'industrial' then non_iou.ind_rate
            when rtk.rate_type = 'residential' then non_iou.res_rate
            end AS rate

        FROM non_iou_df non_iou, rate_type_key rtk
    ),



    COMBINED_DATA AS (
		S

,zip,households,moe_households,households_less_than_$10k,moe_households_less_than_$10k,households_$10k_to_$14k,moe_households_$10k_to_$14k,households_$15k_to_$24k,moe_households_$15k_to_$24k,households_$25k_to_$34k,...,n20_49,n50_99,n100_249,n250_499,n500_999,n1000,service_type,ownership,rate_type_index,rate
0,1002,9715,427,9.7,3.2,5.5,2.0,9.2,3.6,5.9,...,145.0,25.0,0.0,0.0,0.0,0.0,Delivery,Investor Owned,2,0.142767
1,1005,1688,185,1.0,1.6,4.4,5.6,4.1,5.3,4.0,...,13.0,3.0,0.0,0.0,0.0,0.0,Delivery,Investor Owned,2,0.142767
2,1007,6429,359,2.7,1.5,2.9,1.6,6.2,3.1,4.6,...,36.0,0.0,3.0,0.0,0.0,0.0,Delivery,Investor Owned,2,0.142767
3,1009,367,151,10.6,12.7,4.6,7.8,14.2,21.2,9.5,...,0.0,0.0,0.0,0.0,0.0,0.0,Delivery,Investor Owned,2,0.142767
4,1010,1479,110,2.6,2.2,3.9,4.4,5.8,3.4,5.3,...,0.0,0.0,0.0,0.0,0.0,0.0,Delivery,Investor Owned,2,0.142767


In [ ]:
""

''